# TrOCR Fine-Tuning for Maasin Civil Registry (Colab / free T4)

Two-stage fine-tuning of `microsoft/trocr-base-handwritten`:
1. **Stage 1** - train on the ~60k synthetic dataset (learns the domain).
2. **Stage 2** - adapt on real handwriting (optional, run when you have it).

**Free-tier friendly:**
- Checkpoints are saved to your Google Drive **every epoch**.
- If the Colab session disconnects, just re-run the cells - training
  **auto-resumes from the last checkpoint** on Drive.
- Accuracy is measured **before** (base model) and **after** fine-tuning so
  you can see the improvement.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.

In [ ]:
# 1. Confirm a GPU is attached
!nvidia-smi

In [ ]:
# 2. Install dependencies
!pip -q install transformers==4.38.2 sentencepiece evaluate jiwer accelerate

In [ ]:
# 3. Mount Google Drive (checkpoints + final model are stored here so they
#    survive a session disconnect).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Unzip the dataset into /content/data
#    On your PC, zip 'dataset/splits' and upload it to Drive, then set the path.
DATA_ZIP = '/content/drive/MyDrive/splits.zip'

import os, zipfile
os.makedirs('/content/data', exist_ok=True)
with zipfile.ZipFile(DATA_ZIP) as z:
    z.extractall('/content/data')

for root, dirs, files in os.walk('/content/data'):
    if root.replace('/content/data', '').count('/') <= 1:
        print(root, '->', len(files), 'files')

In [ ]:
# 5. CONFIG - adjust paths/hyperparameters here
from pathlib import Path

DATA_DIR = Path('/content/data/splits')
TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR   = DATA_DIR / 'val'
TEST_DIR  = DATA_DIR / 'test'
LABELS_CSV = DATA_DIR / 'labels.csv'

MODEL_NAME = 'microsoft/trocr-base-handwritten'

# Everything below is on Drive so it persists across disconnects.
DRIVE_ROOT  = '/content/drive/MyDrive/trocr_maasin'
CKPT_DIR    = DRIVE_ROOT + '_stage1_ckpt'      # epoch checkpoints (for resume)
FINAL_DIR   = DRIVE_ROOT + '_stage1_final'     # best model, saved at the end
BASELINE_JSON = DRIVE_ROOT + '_baseline.json'  # 'before' accuracy, saved once

MAX_TARGET_LENGTH = 48
BATCH_SIZE = 8
GRAD_ACCUM = 2
EPOCHS = 4
LR = 5e-5
VAL_SUBSET = 1500          # cap val so generate-based eval stays fast
print('Config set. Checkpoints ->', CKPT_DIR)

In [ ]:
# 6. Imports, label loading, helpers, and the Dataset class
import csv, json, random, torch
from PIL import Image
from torch.utils.data import Dataset

def load_labels(csv_path):
    m = {}
    with open(csv_path, encoding='utf-8') as f:
        for row in csv.reader(f):
            if len(row) >= 2 and row[0].lower() != 'filename':
                m[row[0]] = row[1]
    return m

def pct(x):
    return f'{x * 100:.2f}%'

LABEL_MAP = load_labels(LABELS_CSV)
print('labels loaded:', len(LABEL_MAP))

class TrocrDataset(Dataset):
    def __init__(self, img_dir, label_map, processor, max_len, limit=None):
        paths = sorted(p for p in Path(img_dir).glob('*.png') if p.name in label_map)
        if limit and len(paths) > limit:
            random.Random(42).shuffle(paths)
            paths = paths[:limit]
        self.paths = paths
        self.label_map = label_map
        self.processor = processor
        self.max_len = max_len

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        path = self.paths[i]
        image = Image.open(path).convert('RGB')
        pixel_values = self.processor(image, return_tensors='pt').pixel_values[0]
        label = self.label_map[path.name]
        ids = self.processor.tokenizer(
            label, padding='max_length', max_length=self.max_len, truncation=True
        ).input_ids
        pad_id = self.processor.tokenizer.pad_token_id
        ids = [tok if tok != pad_id else -100 for tok in ids]
        return {'pixel_values': pixel_values, 'labels': torch.tensor(ids)}

In [ ]:
# 7. Load processor + model and set the generation tokens
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.max_length = MAX_TARGET_LENGTH
model.config.num_beams = 1
print('model ready')

In [ ]:
# 8. Build datasets + metric (CER + exact-match accuracy)
import evaluate

train_ds = TrocrDataset(TRAIN_DIR, LABEL_MAP, processor, MAX_TARGET_LENGTH)
val_ds   = TrocrDataset(VAL_DIR, LABEL_MAP, processor, MAX_TARGET_LENGTH, limit=VAL_SUBSET)
print('train:', len(train_ds), ' val:', len(val_ds))

cer_metric = evaluate.load('cer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    labels_ids = pred.label_ids
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    exact = sum(p == l for p, l in zip(pred_str, label_str)) / max(1, len(label_str))
    return {'cer': cer, 'exact_match': exact}

In [ ]:
# 9. Build the Trainer. output_dir is on DRIVE and we save EVERY EPOCH,
#    so checkpoints survive a disconnect and we can resume.
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator

args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    evaluation_strategy='epoch',
    save_strategy='epoch',          # checkpoint every epoch
    save_total_limit=2,             # keep last 2 checkpoints on Drive
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics,
)
print('trainer ready')

In [ ]:
# 10. BEFORE fine-tuning: measure the base model's accuracy on val.
#     Saved to Drive so the comparison still works after a disconnect/resume.
if os.path.exists(BASELINE_JSON):
    before = json.load(open(BASELINE_JSON))
    print('Loaded saved baseline:', before)
else:
    print('Evaluating base model (before fine-tuning)... this runs generation, please wait.')
    res = trainer.evaluate(val_ds)
    before = {'cer': res['eval_cer'], 'exact_match': res['eval_exact_match']}
    json.dump(before, open(BASELINE_JSON, 'w'), indent=2)
    print('Baseline (before):  CER', pct(before['cer']), ' exact-match', pct(before['exact_match']))

In [ ]:
# 11. STAGE 1 - train on synthetic, auto-resuming if a checkpoint exists.
from transformers.trainer_utils import get_last_checkpoint

last_ckpt = None
if os.path.isdir(CKPT_DIR):
    last_ckpt = get_last_checkpoint(CKPT_DIR)
if last_ckpt:
    print('Resuming from checkpoint:', last_ckpt)
else:
    print('Starting fresh (no checkpoint found).')

trainer.train(resume_from_checkpoint=last_ckpt)

In [ ]:
# 12. Save the best Stage-1 model to Drive
model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print('saved to', FINAL_DIR)

In [ ]:
# 13. AFTER fine-tuning: measure accuracy and show BEFORE -> AFTER
res_after = trainer.evaluate(val_ds)
after = {'cer': res_after['eval_cer'], 'exact_match': res_after['eval_exact_match']}

print('=' * 52)
print('STAGE 1 RESULTS (validation set)')
print('=' * 52)
print(f'{"metric":<22}{"before":>12}{"after":>12}')
print('-' * 52)
print(f'{"Character Error Rate":<22}{pct(before["cer"]):>12}{pct(after["cer"]):>12}   (lower is better)')
print(f'{"Exact-match accuracy":<22}{pct(before["exact_match"]):>12}{pct(after["exact_match"]):>12}   (higher is better)')
print('-' * 52)
cer_drop = (before['cer'] - after['cer']) * 100
acc_gain = (after['exact_match'] - before['exact_match']) * 100
print(f'CER improved by {cer_drop:.2f} points; exact-match improved by {acc_gain:.2f} points.')

## Stage 2 - adapt on real handwriting (run later)

When you have real samples, put them in a folder with their own
`train/`, `val/`, `test/` + `labels.csv`, zip + upload, then set `REAL_DIR`
and `RUN_STAGE2 = True` below. This continues from the Stage-1 model, also
checkpoints every epoch to Drive, and reports before/after on the REAL data.

In [ ]:
# 14. STAGE 2 (optional)
RUN_STAGE2 = False   # set True when real data is ready

if RUN_STAGE2:
    from torch.utils.data import ConcatDataset
    from transformers.trainer_utils import get_last_checkpoint

    REAL_DIR = Path('/content/data/real_splits')   # adjust
    S2_CKPT = DRIVE_ROOT + '_stage2_ckpt'
    S2_FINAL = DRIVE_ROOT + '_stage2_final'
    S2_BASELINE = DRIVE_ROOT + '_stage2_baseline.json'

    real_labels = load_labels(REAL_DIR / 'labels.csv')
    real_train = TrocrDataset(REAL_DIR / 'train', real_labels, processor, MAX_TARGET_LENGTH)
    real_val   = TrocrDataset(REAL_DIR / 'val', real_labels, processor, MAX_TARGET_LENGTH)

    # mix in ~20% synthetic so the model doesn't forget the domain
    synth_mix = TrocrDataset(TRAIN_DIR, LABEL_MAP, processor, MAX_TARGET_LENGTH,
                             limit=max(1, len(real_train) // 4))
    stage2_train = ConcatDataset([real_train, synth_mix])

    args2 = Seq2SeqTrainingArguments(
        output_dir=S2_CKPT,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=2e-5,
        num_train_epochs=6,
        fp16=True,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=2,
        logging_steps=20,
        load_best_model_at_end=True,
        metric_for_best_model='cer',
        greater_is_better=False,
        report_to='none',
    )
    trainer2 = Seq2SeqTrainer(
        model=model, args=args2,
        train_dataset=stage2_train, eval_dataset=real_val,
        data_collator=default_data_collator,
        tokenizer=processor.feature_extractor,
        compute_metrics=compute_metrics,
    )

    # before (on REAL val) - this is the Stage-1 model's real-world accuracy
    if os.path.exists(S2_BASELINE):
        s2_before = json.load(open(S2_BASELINE))
    else:
        r = trainer2.evaluate(real_val)
        s2_before = {'cer': r['eval_cer'], 'exact_match': r['eval_exact_match']}
        json.dump(s2_before, open(S2_BASELINE, 'w'), indent=2)
    print('Stage-2 BEFORE (real val):  CER', pct(s2_before['cer']),
          ' exact', pct(s2_before['exact_match']))

    s2_last = get_last_checkpoint(S2_CKPT) if os.path.isdir(S2_CKPT) else None
    trainer2.train(resume_from_checkpoint=s2_last)
    model.save_pretrained(S2_FINAL)
    processor.save_pretrained(S2_FINAL)

    r2 = trainer2.evaluate(real_val)
    s2_after = {'cer': r2['eval_cer'], 'exact_match': r2['eval_exact_match']}
    print('Stage-2 AFTER  (real val):  CER', pct(s2_after['cer']),
          ' exact', pct(s2_after['exact_match']))
else:
    print('Stage 2 skipped (set RUN_STAGE2 = True when real data is ready).')

In [ ]:
# 15. Final evaluation on the held-out TEST set (if present)
test_paths = list(TEST_DIR.glob('*.png')) if TEST_DIR.exists() else []
if test_paths:
    test_ds = TrocrDataset(TEST_DIR, LABEL_MAP, processor, MAX_TARGET_LENGTH)
    m = trainer.evaluate(test_ds)
    print('TEST  CER', pct(m['eval_cer']), ' exact-match', pct(m['eval_exact_match']))
else:
    print('No test images found. (Synthetic-only runs have no test set;')
    print(' the real handwriting becomes your test set in Stage 2.)')

In [ ]:
# 16. Quick sanity check - predict a few validation images
model.eval().to('cuda')
for path in val_ds.paths[:8]:
    image = Image.open(path).convert('RGB')
    pv = processor(image, return_tensors='pt').pixel_values.to('cuda')
    with torch.no_grad():
        out = model.generate(pv, max_length=MAX_TARGET_LENGTH)
    pred = processor.batch_decode(out, skip_special_tokens=True)[0]
    print(f'{path.name:>16}  pred: {pred!r}   truth: {LABEL_MAP[path.name]!r}')